In [ ]:
%load_ext autoreload 
%autoreload 2

In [ ]:
import polars as pl
import datetime as dt

import numpy as np
from functools import reduce


### Before we get started, let's prototype the functional behaviour we want on a simpler problem

In [ ]:
from typing import Self

class Operation:
    def __init__(self, nodes: list[Self]):
        self.nodes = nodes

    def operate(self, x):
        current = x
        for node in self.nodes:
            current = node.operate(current)
        return current

    def compose(self, other: Self):
        return Operation(self.nodes + other.nodes)

    def __sum__(self, other: Self):
        return self.compose(other)

class PrimitiveOperation(Operation):

    def __init__(self):
        self.nodes = [self]


class Add2(PrimitiveOperation):

    def operate(self, x):
        return x + 2


In [ ]:
add2 = Add2()
add4 = add2.compose(add2)

In [ ]:
add4.operate(1)

### Let's start prototyping a new PoMap

In [ ]:
# # First, a sketch, a Pomap is defined by a 'dimension' and a set of expected labels     

# # TODO - allow better labels by adding a reference column instead

# class _Pomap:

#     # A PoMap is defined by a 'dimension' and a set of labels belonging to that dimension
#     def __init__(self, nodes: list[Self], name: str):

#         self.name = name
#         self._nodes = nodes

#         # Implement some standardised naming for the subclasses to use
#         self._train_column_name = lambda label:  f'train({self.name}={label})'
#         self._test_column_name = lambda label: f'test({self.name}={label})'
#         self._validate_column_name = lambda label: f'validate({self.name}={label})'

#     def __repr__(self):
#         return self.name


#     def __getitem__(self, arg: str):
#         for node in self._nodes:
#             if node.name == 'arg':
#                 return node
#         raise ValueError(f'Pomap has no node {arg}')
                
# # -=-=-=-=-=-=-=--=-=-==-=-=-=-=-=--=-=-=-=-=-=-=-=-=-=-=-=-==-=-=-=-=-=-=-=-=-=--=-
# #   After this, things get interesting, since we start to deal with how the pomap actually behaves
# # COMPOSITION


#     def product(self, other: Self):
#         # This composition assumes that ONLY the product operation is possible, not the sum.

#         overlapping_names = set(self._nodes).intersection(other._nodes) 
#         assert overlapping_names == set(), f"Cannot compose two Pomaps with overlapping names. Found {overlapping_names} in common"
#         return _Pomap(nodes=self._nodes + other._nodes, name=f'{self.name} + {other.name}')

#     @property
#     def labels(self) -> pl.DataFrame:
#         # TODO this will have to change if we introduce a product operation
#         # (or any other composition)
#         # Since we will need to recurse through the syntax tree and 
#         # pluck out the labels of all the child nodes

#         leaf_nodes = self._find_leaf_nodes(self)               
#         leaf_labels = [node.labels for node in leaf_nodes]

#         df = reduce(lambda left, right: left.to_frame().join(right.to_frame(), how='cross'),  leaf_labels)

#         return df


#     @staticmethod
#     def _find_leaf_nodes(node: _Pomap):
#         if (len(node._nodes) == 1) and (node._nodes[0] is node):
#             return [node]
        
#         leaf_nodes = []
#         for child in node._nodes:
#             leaf_nodes.extend(_Pomap._find_leaf_nodes(child))
        
#         return leaf_nodes

#     # Need to implement these in terms of the composed logic
#     def label_rows_as_train(self, df: pl.DataFrame, label: dict) -> pl.DataFrame:
        
#         node_columns = []
#         for node in self._nodes:
#             node_sub_label = label[node.name]
#             df = node.label_rows_as_train(df, node_sub_label)
#             node_columns.append(node._train_column_name(node_sub_label))

#         df = df.with_columns(node_trains = pl.concat_list(node_columns))
#         df = df.with_columns(
#             pl.col('node_trains').list.all()
#             .alias(self._train_column_name(label))
#             )
#         df = df.drop('node_trains', *node_columns)

#         return df

#     def label_rows_as_test(self, df: pl.DataFrame, label: dict) -> pl.DataFrame:
#         pass

#     def label_rows_as_validate(self, df: pl.DataFrame, label: dict) -> pl.DataFrame:
#         pass 

# #### Model Interface
#     def label_to_train(self, df: pl.DataFrame, label: dict):
#         df = self.label_rows_as_train(df, label)
#         df = df.filter(
#             self.train_column_name(label)
#         )
#         df = df.drop(
#             self.train_column_name(label)
#         )

#         return df

In [ ]:
# class Pomap(_Pomap):

#     def __init__(self, name: str):
#         super().__init__(nodes=[self], name=name)

#     @property
#     def labels(self, other: Self) -> pl.Series:
#         raise NotImplementedError

#     # These three (train, test, validate) functions define the behaviour of the PoMap.
#     # E.g, is it a cross validation, is it categorical, etc.
#     # See below for an example of a reasonably complex example.
#     def label_rows_as_train(self, df: pl.DataFrame, label: dict) -> pl.DataFrame:
#         raise NotImplementedError

#     # There has to be a separate one for test and validation, because
#     # train and test data must be distinct.
#     def label_rows_as_test(self, df: pl.DataFrame, label: dict) -> pl.DataFrame:
#         raise NotImplementedError

#     # .... as above
#     def label_rows_as_validate(self, df: pl.DataFrame, label: dict) -> pl.DataFrame:
#         raise NotImplementedError

In [ ]:
# class CategoricalPomap(Pomap):

#     def __init__(self, column: str, labels: list):
#         super().__init__(name=f"Categorical {column}: {labels}")
#         self._column = column
#         self._labels = labels

#     @property
#     def labels(self) -> pl.Series:
#         return pl.Series(values=self._labels, name=self.name)

#     def _label_rows_as(self, df, label, column_name) -> pl.DataFrame:

#         df =  df.with_columns(
#             (pl.col(self._column) == label).alias(column_name)
#         )

#         return df
    
#     def label_rows_as_train(self, df: pl.DataFrame, label) -> pl.DataFrame:
#         return self._label_rows_as(df, label, self._train_column_name(label))

#     def label_rows_as_test(self, df: pl.DataFrame, label) -> pl.DataFrame:
#         return self._label_rows_as(df, label, self._test_column_name(label))

#     def label_rows_as_validate(self, df: pl.DataFrame, label) -> pl.DataFrame:
#         return self._label_rows_as(df, label, self._validate_column_name(label))
        
        

### Load the Logic from the repo instead

In [ ]:
from pomap_v2.categorical_pomap import CategoricalPomap
from pomap_v2.cross_validation_pomap import RandomisedCrossValidationPoMap

#### Example time

In [ ]:
example_df = pl.from_dict(
    {'cat1': ['a',  'b',],
    'cat2': ['c', 'd', ]}
)

timestamps = pl.datetime_range(dt.datetime(2025, 1, 1),
                               dt.datetime(2025, 1, 7),
                               interval='1h',
                               eager=True
                               ).rename('ts').to_frame()

example_df = example_df.join(timestamps, how='cross')
example_df = example_df.with_columns(
    feature = np.random.normal(0, 1, example_df.shape[0])
)

example_df = example_df.with_columns(hour=pl.col('ts').dt.hour())

In [ ]:
pmap_cat1 = CategoricalPomap(column='cat1', labels=['a', 'b'])

In [ ]:
pmap_cat2 = CategoricalPomap(column='cat2', labels=['c', 'd'])

In [ ]:
composed = pmap_cat1.product(pmap_cat2)

In [ ]:
composed.labels

In [ ]:
# Test labeling logic

In [ ]:
pmap_cat1.label_rows_as_train(example_df, label={'cat1': 'a'}).head()

In [ ]:
composed = pmap_cat1.product(pmap_cat2)
composed.label_rows_as_train(example_df, label={'cat1': 'a', 'cat2': 'c'}).head()

### Prototype the model behaviour

In [ ]:
import polars as pl
import statsmodels.formula.api as smf

from pomap_v2.pomap import Pomap

from typing import Callable

In [ ]:
#### Toy example

N = 1000
x = np.random.normal(0, 1, size=N)
eps = np.random.normal(scale=0.1, size=x.shape[0])


df = pl.from_dict({'x': x, 'eps': eps})

df = df.with_columns(
    cat1=np.random.choice(['a', 'b'], size=df.shape[0]),
    cat2=np.random.choice(['c', 'd'], size=df.shape[0])
                    )

df = df.with_columns(slope = pl.col('cat1').replace_strict({'a': 5, 'b': 50}),
                     intercept = pl.col('cat2').replace_strict({'c': 0, 'd': -100})
                    )

df = df.with_columns(y=pl.col('slope')*pl.col('x') + pl.col('intercept') + pl.col('eps'))

# We don't get to see eps or the model parameters
train_df = df.select('x', 'y', 'cat1', 'cat2')

In [ ]:
constructor = lambda train_data: smf.ols(formula='y~x', data=train_data.to_pandas())

pomap = CategoricalPomap('cat1', ['a', 'b']).product(
    CategoricalPomap('cat2', ['c',])
)

In [ ]:
# Need to implement this in PoMap, so that it expands on composition
from collections import namedtuple
Label = namedtuple('Label', ['cat1', 'cat2'])

In [ ]:
class LiftedSmf:

    def __init__(self, model_constructor, pomap: Pomap):
        self._constructor = model_constructor
        self.pomap = pomap
        self.models = {}
        self.Label = namedtuple('Label', self.pomap.labels.columns)

    def fit(self, train_data: pl.DataFrame):

        # TODO replace with iter_labels
        for label in self.pomap.labels.iter_rows(named=True):
            sub = self.pomap.label_to_train(df, label=label)
            model = self._constructor(sub).fit()
        
            self.models[self.Label(**label)] = model

  
    # TODO I think this one can be cleaned up into a single 'apply' function
    def predict(self, test_data: pl.DataFrame):

        test_data = test_data.with_row_index(name='__row_index')

        subs = []
        for label in self.pomap.labels.iter_rows(named=True):
            sub = self.pomap.label_to_test(test_data, label=label)

            model = self.models[self.Label(**label)]

            predictions = model.predict(
                sub.to_pandas()
                ).rename('prediction')
            
            predictions = pl.from_pandas(predictions)
            
            sub = sub.with_columns(predictions).select('prediction', '__row_index')
            subs.append(sub)

        test_data = test_data.join(pl.concat(subs), on='__row_index', how='left').drop('__row_index')

        return test_data

In [ ]:
ls = LiftedSmf(constructor, pomap)
ls.fit(train_df)

In [ ]:
predictions = ls.predict(train_df)
predictions.filter(pl.col('cat2') == 'd')['prediction'].is_null().mean()

In [ ]:
# Now test whether we can 'surgery' in a second model which trains unconditionally on cat2='d'
pomap2 = CategoricalPomap('cat2', ['d'])
ls2 = LiftedSmf(constructor, pomap2)
ls2.fit(train_df)

In [ ]:
predictions2 = ls2.predict(train_df)
predictions2.filter(
    pl.col('cat2') == 'd'
)['prediction'].is_null().mean()

In [ ]:
from dataclasses import dataclass